In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

from tfm_airquality import config
from tfm_airquality.load import load_airquality_data
from tfm_airquality.clean import clean
from tfm_airquality.clean import gap_runs
from tfm_airquality import eda
import matplotlib.pyplot as plt
import seaborn as sns

df, informe_huecos = clean(load_airquality_data())
print(df.shape)
Y = 'NO2(GT)'
X = ['CO(GT)', 'NOx(GT)',
     'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)',
     'T', 'RH', 'AH']
variables = X + [Y]

# Estación 4 — Análisis descriptivo

## Pregunta 1: ¿qué variables anticipan el NO₂?

La correlación contemporánea entre un sensor y el NO₂ no mide capacidad
predictiva. Este proyecto predice a 24 y 48 horas, así que lo relevante es la
relación entre el valor actual de cada variable y el NO₂ futuro.

In [ ]:
tabla = eda.correlacion_por_horizonte(df, variables=X)

fig = plt.figure(figsize=(8, 6))
sns.heatmap(tabla, annot=True, cmap='coolwarm', center=0)
plt.title('Correlación con NO₂ según horizonte')
plt.show()

eda.guardar_figura(fig, 'correlacion_por_horizonte')

## Pregunta 2: ¿se solapan las variables entre sí?

Dos motivos para mirarlo. Primero, la multicolinealidad afecta a la estabilidad
de los modelos lineales y reparte de forma arbitraria la importancia entre
variables redundantes, lo que complica la interpretabilidad. Segundo, permite
cerrar la sospecha sobre `C6H6(GT)`.

In [ ]:
fig = plt.figure(figsize=(10, 8))
sns.heatmap(df[variables].corr().round(2), annot=True, cmap='coolwarm',
            center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlación entre variables')
eda.guardar_figura(fig, 'matriz_correlacion')
plt.show()

In [ ]:
print(df['C6H6(GT)'].nunique())
print(df['PT08.S2(NMHC)'].nunique())

In [ ]:
perfil_horario = df.groupby(df.index.hour)[Y].mean()
perfil_semanal = df.groupby(df.index.dayofweek)[Y].mean()
print(perfil_horario)
print(perfil_semanal)

In [ ]:
autocorr = pd.Series({lag: df[Y].autocorr(lag) for lag in range(1, 193)})
print(autocorr)
for lag in [1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168, 192]:
    print(lag, round(df[Y].autocorr(lag), 3))

In [ ]:
observado = df[Y].dropna()
episodios = observado.where(observado <= 200)
rachas = gap_runs(episodios)

print('episodios:', len(rachas))
print('horas totales:', int(rachas['horas'].sum()), '(deben ser 386)')
print('duración media:', round(rachas['horas'].mean(), 1), 'h')
print('duración máxima:', int(rachas['horas'].max()), 'h')
print(rachas['horas'].value_counts().sort_index().to_dict())

In [ ]:
mes = df.index.to_period('M')

corr = pd.DataFrame({
    s: df.groupby(mes).apply(lambda g: g[s].corr(g[Y]), include_groups=False)
    for s in config.SENSOR_COLUMNS
}).round(3)
print(corr)

ratios = pd.DataFrame({
    s: (df.groupby(mes)[Y].mean() / df.groupby(mes)[s].mean()) * 1000
    for s in config.SENSOR_COLUMNS
}).round(1)
print(ratios)

In [ ]:
fig = eda.plot_deriva_sensores(df)
eda.guardar_figura(fig, 'deriva_sensores')

In [ ]:
fig = eda.plot_simpson(df)
eda.guardar_figura(fig, 'simpson_pt08s4')

In [ ]:
fig = eda.plot_perfiles_temporales(df)
eda.guardar_figura(fig, 'perfiles_temporales')

In [ ]:
fig = eda.plot_autocorrelacion(df)
eda.guardar_figura(fig, 'autocorrelacion')